# Model Parameters Optimization

## Getting Data + Target encoding

In [20]:
import pandas as pd

data = pd.read_csv('Data/trainVal.csv', index_col=False)
X_trainval, y_trainval = data.drop('Air_Quality', axis=1), data['Air_Quality']

# map = {'Hazardous': 0, 'Poor': 1, 'Moderate': 2, 'Good': 3} # Ordinal Encoding (as data is Ordinal) / other approach
# y_trainval = y_trainval.map(map)

y_trainval = pd.get_dummies(y_trainval) # one hot encoding - no bias, we don't assume anything about data

y_trainval

,Good,Hazardous,Moderate,Poor
0,True,False,False,False
1,True,False,False,False
2,True,False,False,False
3,False,False,False,True
4,False,True,False,False
...,...,...,...,...
3995,False,False,True,False
3996,False,False,True,False
3997,False,False,False,True
3998,False,True,False,False


In [21]:

from utils.training import get_dataSet

dataset = get_dataSet(X_trainval, y_trainval)

type(dataset)

torch.utils.data.dataset.TensorDataset

# Optuna Study

In [30]:
import os
import shutil
from torch.utils.tensorboard import SummaryWriter

writer_path = os.path.join('runs', 'optuna')
if os.path.exists(writer_path):
    print(f"Cleaning existing logs at {writer_path}...")
    shutil.rmtree(writer_path)

os.makedirs(writer_path, exist_ok=True)
writer = SummaryWriter(writer_path)

Cleaning existing logs at runs/optuna...


# Objective
**Prompt**:
```
For 4000 instance trianval dataset, suggest best ranges for this parameters.
The network is optimized with 5 fold cross validation by mean validation loss score.

Params to optimize ...
```

| Parameter | Suggested Range | Reasoning |
| :--- | :--- | :--- |
| **`lr`** | `1e-4` to `1e-2` | `5e-1` (0.5) is extremely high and will likely cause the loss to explode. |
| **`batch_size`** | `16` to `128` | With 4,000 samples, a batch of 1 is too noisy; 256 is nearly 10% of your training fold. |
| **`beta1`** | `0.85` to `0.95` | This controls momentum. The default is 0.9. |
| **`beta2`** | `0.99` to `0.9999` | This controls the moving average of squared gradients. |

The goal is to find best parameters for **100** Epochs

In [31]:
import numpy as np
import optuna
from functools import partial
from torch import nn
from torch import optim
from torch.utils.data import TensorDataset
from sklearn.model_selection import KFold

from utils.training import *
from utils.MLPClassifier import MLPClassifier

# TODO: imports

N_EPOCHS = 1
def objective_nn(trial:optuna.Trial, dataset:TensorDataset, input_dim:int, output_dim:int, writer:SummaryWriter):
    global N_EPOCHS

    # Network Params
    # Using step=16 or 32 helps keep dimensions hardware-friendly
    hidden_dim = trial.suggest_int('hidden_dim', 32, 512, step=32)
    n_hidden = trial.suggest_int('n_hidden', 1, 5)

    # Optimizer Params
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])

    # If using Adam, betas are usually stable.
    # Only tune them if you have a specific reason to deviate from defaults.
    beta1 = trial.suggest_float('beta1', 0.8, 0.99)
    beta2 = trial.suggest_float('beta2', 0.99, 0.9999)

    # Scheduler Params
    factor = trial.suggest_float('factor', 0.1, 0.5)
    patience = trial.suggest_int('patience', 3, 7)


    fold_loss = []
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_trainval)):

        # Loaders
        train_loader, val_loader = get_train_loaders(dataset, train_idx=train_idx, val_idx=val_idx, batch_size=batch_size)

        # Model, Optimizer, Criterion
        model = MLPClassifier(input_dim=input_dim, output_dim=output_dim, hidden_dim=0, n_hidden=0)
        criterion = nn.CrossEntropyLoss() # nn.MSELoss (better if we choose ordinal encoding)
        optimizer = optim.Adam(lr=lr, params=model.parameters(), betas=(beta1, beta2))
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=factor, patience=patience)

        # Fold Training
        loss = train_one_fold(fold, model, train_loader, val_loader, optimizer, scheduler, criterion, n_epochs=N_EPOCHS, write_model_dir=None, writer=None)
        fold_loss.append(loss)

    avg_loss = np.mean(fold_loss)
    # Writing Params
    writer.add_hparams(
        hparam_dict={
        'trial': trial.number,
        'hidden_dim': hidden_dim,
        'n_hidden': n_hidden,
        'lr': lr,
        'batch_size': batch_size,
        'beta1': beta1,
        'beta2': beta2,
        'factor': factor,
        'patience': patience
    },
    metric_dict={'hparam/mean_loss': avg_loss})
    writer.flush()


    return avg_loss

objective = partial(objective_nn, dataset=dataset, input_dim=X_trainval.shape[1], output_dim=y_trainval.shape[1], writer=writer)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=5, show_progress_bar=True)

[I 2026-04-21 16:28:22,414] A new study created in memory with name: no-name-c1519e7a-489c-40b8-8a36-124ba3540ce0


  0%|          | 0/5 [00:00<?, ?it/s]

/home/franio/Desktop/Uczenie_maszynowe/venv/lib/python3.11/site-packages/torch/nn/init.py:452: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


[I 2026-04-21 16:28:22,760] Trial 0 finished with value: 0.19761662921127007 and parameters: {'hidden_dim': 288, 'n_hidden': 5, 'lr': 0.00032588248089173496, 'batch_size': 128, 'beta1': 0.9567142040816047, 'beta2': 0.9982101081960224, 'factor': 0.3125499414251807, 'patience': 7}. Best is trial 0 with value: 0.19761662921127007.
[I 2026-04-21 16:28:25,034] Trial 1 finished with value: 0.027515389404296875 and parameters: {'hidden_dim': 480, 'n_hidden': 3, 'lr': 0.0001880524225187761, 'batch_size': 16, 'beta1': 0.8954246950266995, 'beta2': 0.9936007674915895, 'factor': 0.4133554966413955, 'patience': 4}. Best is trial 1 with value: 0.027515389404296875.
[I 2026-04-21 16:28:28,220] Trial 2 finished with value: 0.02754938067436218 and parameters: {'hidden_dim': 128, 'n_hidden': 2, 'lr': 0.00015781945190569788, 'batch_size': 16, 'beta1': 0.8729606501559496, 'beta2': 0.998898700836106, 'factor': 0.39192685663962823, 'patience': 3}. Best is trial 1 with value: 0.027515389404296875.
[I 2026-04

In [ ]:
# tensorboard --logdir Lab6_7-project/runs/optuna

In [26]:
import importlib
import utils.training

# After you change my_script.py:
importlib.reload(utils.training)

<module 'utils.training' from '/home/franio/Desktop/Uczenie_maszynowe/Deep_Learning/Lab6_7-project/utils/training.py'>